# Train the Phase 6 models (helmet + plate) on Google Colab

Trains the Phase 6 extension models on a Colab GPU:

- **helmet** — helmet non-use detection (`configs/train.helmet.yaml`, Roboflow helmet-detection_yolov8 v3, CC BY 4.0)
- **plate** — license-plate detector for the optional ANPR module (`configs/train.plate.yaml`, License Plate Recognition v13, CC BY 4.0)

The Phase 1 vehicle detector is already trained and committed
(`weights/detector.pt`, `docs/RESULTS.md`) — no need to re-run it here.

**Before running:** in Colab, open the key/secrets panel (left sidebar) and add
`ROBOFLOW_API_KEY` with your key from https://app.roboflow.com/settings/api.
The key stays in Colab secrets — it is never written to the repo.

In [ ]:
# 1. Clone the repo (Phase 6 branch) and install deps
!git clone -b phase-6-extensions https://github.com/Rakib-codee/Traffic-Violation-Detection.git tvd
%cd tvd
!pip install -q -e ".[dev,data]"

In [ ]:
# 2. Read the Roboflow API key from Colab secrets
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("ROBOFLOW_API_KEY loaded from secrets")

In [ ]:
# 3. Download the helmet dataset (helmet-detection_yolov8 v3, YOLOv8 format)
!python scripts/download_dataset.py \
    --workspace learning-evidence --project helmet-detection_yolov8 --version 3 \
    --format yolov8 --location data/helmet

In [ ]:
# 4. Download the plate dataset (License Plate Recognition v13, YOLOv8 format)
!python scripts/download_dataset.py \
    --workspace roboflow-universe-projects --project license-plate-recognition-rxg4e --version 13 \
    --format yolov8 --location data/plates

In [ ]:
# 5. Sanity-check both datasets
from tvd.detect.data import validate_dataset

for yaml_path in ("data/helmet/data.yaml", "data/plates/data.yaml"):
    i = validate_dataset(yaml_path)
    print(yaml_path, "->", i.n_train, "train /", i.n_val, "val /", i.names)

In [ ]:
# 6. Train the helmet model (20 epochs, CUDA) -> weights/helmet.pt
!python scripts/train_detector.py --train-config configs/train.helmet.yaml --device cuda

In [ ]:
# 7. Train the plate model (20 epochs, CUDA) -> weights/plate.pt
!python scripts/train_detector.py --train-config configs/train.plate.yaml --device cuda

In [ ]:
# 8. Evaluate the helmet model -> results/detection/helmet/mAP.md + metrics.json
!python scripts/eval_detector.py --weights weights/helmet.pt \
    --train-config configs/train.helmet.yaml --device cuda

In [ ]:
# 9. Evaluate the plate model -> results/detection/plate/mAP.md + metrics.json
!python scripts/eval_detector.py --weights weights/plate.pt \
    --train-config configs/train.plate.yaml --device cuda

In [ ]:
# 10. Download the artifacts back to your machine
# (docs/RESULTS.md is overwritten by the last eval, so the per-run mAP.md files
# are the ones to keep — commit them under docs/ back on your machine.)
from google.colab import files

files.download("weights/helmet.pt")
files.download("weights/plate.pt")
files.download("results/detection/helmet/mAP.md")
files.download("results/detection/plate/mAP.md")